In [24]:
from Sequential_Fish.chromatic_abberrations import load_calibration, apply_polynomial_transform_spots,apply_polynomial_transform_to_signal , get_polynomial_features, CALIBRATION_FOLDER
from Sequential_Fish.chromatic_abberrations.calibration import _load_calibration_index
from Sequential_Fish.tools.utils import open_image
test_path = "/media/SSD_floricslimani/Fish_seq/Davide/2024-08-12 - SeqFISH - HeLa - Puro - R2TP1-2_Run7/result_tables/Spots.feather"
calibration_image_path = "/media/SSD_floricslimani/Fish_seq/Davide/Chromatic abberations/Beads-Field_01.tif"

In [25]:
import pandas as pd
Spots = pd.read_feather(test_path)
Spots

,spot_id,cluster_id,drifted_z,drifted_y,drifted_x,intensity,population,detection_id,acquisition_id,drift_z,...,x_shape,z,y,x,cycle,color_id,is_washout,coordinates,in_nucleus,cell_label
0,0,NaN,5,93,1600,8120,free,1,0,0,...,2004,5,93,1600,0,0,False,"[5, 93, 1600]",True,1.0
1,1,NaN,5,123,1625,8751,free,1,0,0,...,2004,5,123,1625,0,0,False,"[5, 123, 1625]",True,1.0
2,2,NaN,5,128,1634,8427,free,1,0,0,...,2004,5,128,1634,0,0,False,"[5, 128, 1634]",True,1.0
3,3,NaN,5,141,610,8403,free,1,0,0,...,2004,5,141,610,0,0,False,"[5, 141, 610]",True,4.0
4,4,NaN,5,157,1650,8633,free,1,0,0,...,2004,5,157,1650,0,0,False,"[5, 157, 1650]",True,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
374065,229359,NaN,37,301,290,4426,free,140,121,0,...,2004,37,301,290,13,1,True,"[37, 301, 290]",False,0.0
374066,229360,NaN,14,1902,1854,4059,free,140,121,0,...,2004,14,1902,1854,13,1,True,"[14, 1902, 1854]",False,0.0
374067,264241,NaN,15,1549,595,7244,free,168,122,0,...,2004,15,1549,595,13,1,True,"[15, 1549, 595]",False,32.0
374068,312150,NaN,17,505,864,6322,free,196,123,0,...,2004,17,505,864,13,1,True,"[17, 505, 864]",True,11.0


In [26]:
calibration = load_calibration(555,640)
calibration

{'x_fit': LinearRegression(),
 'y_fit': LinearRegression(),
 'z_fit': LinearRegression(),
 'polynomial_features': PolynomialFeatures(),
 'polynomial_features_inv': PolynomialFeatures(),
 'x_inv_fit': LinearRegression(),
 'y_inv_fit': LinearRegression(),
 'z_inv_fit': LinearRegression(),
 'voxel_size': (200, 97, 97),
 'degree': 2,
 'reference_wavelength': 555,
 'corrected_wavelength': 640,
 'timestamp': '07_07_2025_11_33_58'}

# Check on calibration image : 
--> Check that model are still correctly fitted afte saving with joblib

In [27]:
calibration_image = open_image(calibration_image_path)
calibration_image.shape

(47, 1981, 2004, 3)

In [28]:
corrected_image = apply_polynomial_transform_to_signal(
    image= calibration_image[...,2],
    poly= calibration['polynomial_features_inv'],
    model_x= calibration['x_inv_fit'],
    model_y= calibration['y_inv_fit'],
    model_z= calibration['z_inv_fit'],
    voxel_size=calibration['voxel_size']
)

corrected_image_2d = apply_polynomial_transform_to_signal(
    image= calibration_image[...,2],
    poly= calibration['polynomial_features_inv'],
    model_x= calibration['x_inv_fit'],
    model_y= calibration['y_inv_fit'],
    voxel_size=calibration['voxel_size']
)

(186586428, 3)


In [29]:
import napari, os
os.environ["QT_QPA_PLATFORM"] = "xcb"

voxel_size = calibration['voxel_size']

Viewer = napari.Viewer()
Viewer.add_image(calibration_image[...,1], scale= voxel_size, name= "reference", colormap= "green", blending= "additive")
Viewer.add_image(calibration_image[...,2], scale= voxel_size, name= "abeeration", colormap= "red", blending= "additive")
Viewer.add_image(corrected_image, scale= voxel_size, name= "corrected_image", colormap= "blue", blending= "additive")
Viewer.add_image(corrected_image_2d, scale= voxel_size, name= "corrected_image_2d", colormap= "blue", blending= "additive")

napari.run()

## Check on spots

In [35]:
def convert_str_coordinates_to_zyx_columns(coordinates_series : pd.Series) :
    coordinates_series = coordinates_series.str.replace('(','').str.replace(')','').str.split(',')
    res_df = pd.DataFrame(coordinates_series)
    res_df['z'] = res_df['coordinates'].apply(lambda x : int(x[0])).astype(int)
    res_df['y'] = res_df['coordinates'].apply(lambda x : int(x[2])).astype(int)
    res_df['x'] = res_df['coordinates'].apply(lambda x : int(x[1])).astype(int)

    return res_df


In [36]:
import pandas as pd
import numpy as np
spots_path = "/media/SSD_floricslimani/Fish_seq/Davide/Chromatic abberations/"
ref_spots = pd.read_csv(spots_path + "/Beads-Field_01_channel1.csv", sep= ";")['coordinates'] #555nm
abb_spots = pd.read_csv(spots_path + "/Beads-Field_01_channel2.csv", sep= ";")['coordinates'] #640nm
ref_spots = convert_str_coordinates_to_zyx_columns(ref_spots)
abb_spots = convert_str_coordinates_to_zyx_columns(abb_spots)
ref_spots

,coordinates,z,y,x
0,"[17, 106, 1888]",17,1888,106
1,"[18, 65, 1819]",18,1819,65
2,"[18, 77, 1738]",18,1738,77
3,"[18, 1944, 23]",18,23,1944
4,"[19, 59, 1768]",19,1768,59
...,...,...,...,...
527,"[32, 1079, 1031]",32,1031,1079
528,"[45, 1339, 1775]",45,1775,1339
529,"[46, 339, 1249]",46,1249,339
530,"[46, 781, 1796]",46,1796,781


In [37]:
abb_coordinates = np.array(list(
    zip(abb_spots['z'],abb_spots['y'],abb_spots['x'],)
)).astype(int)

ref_coordinates = np.array(list(
    zip(ref_spots['z'],ref_spots['y'],ref_spots['x'],)
)).astype(int)

ref_coordinates.shape

(532, 3)

In [42]:
spots_3d_corrected = apply_polynomial_transform_spots(
    abb_coordinates,
    poly=calibration['polynomial_features_inv'],
    model_x=calibration['x_inv_fit'],
    model_y=calibration['y_inv_fit'],
    model_z=calibration['z_inv_fit'],
    voxel_size=calibration['voxel_size']
)

spots_2d_corrected = apply_polynomial_transform_spots(
    abb_coordinates,
    poly=calibration['polynomial_features'],
    model_x=calibration['x_fit'],
    model_y=calibration['y_fit'],
    voxel_size=calibration['voxel_size']
)

**#TODO**

* Model et model inv corrige les spots dans la même direction ? (et la mauvaise pour les spots, donc la bonne pour le signal) (model inv est toujours sauvegardé ??)

* 2d correction actually put all spots on maximum (or minimum ? anyway topmost) plane

In [44]:
import napari, os
os.environ["QT_QPA_PLATFORM"] = "xcb"

voxel_size = calibration['voxel_size']

Viewer = napari.Viewer()
Viewer.add_image(calibration_image[...,1], scale= voxel_size, name= "reference", colormap= "green", blending= "additive")
Viewer.add_image(calibration_image[...,2], scale= voxel_size, name= "abeeration", colormap= "red", blending= "additive")
Viewer.add_image(corrected_image, scale= voxel_size, name= "corrected_image", colormap= "blue", blending= "additive")
Viewer.add_image(corrected_image_2d, scale= voxel_size, name= "corrected_image_2d", colormap= "blue", blending= "additive")

Viewer.add_points(ref_coordinates, scale= voxel_size, name= "ref_spots", blending="additive", face_color= "transparent")
Viewer.add_points(spots_2d_corrected, scale= voxel_size, name= "spots_2d_corrected", blending="additive", face_color= "transparent")
Viewer.add_points(spots_3d_corrected, scale= voxel_size, name= "spots_3d_corrected", blending="additive", face_color= "transparent")
Viewer.add_points(abb_coordinates, scale= voxel_size, name= "abb_coordinates", blending="additive", face_color= "transparent")

napari.run()